# Software Defect Prediction Model
### Based on NASA JM1 Dataset & Radon Metrics

---

## Project Overview
This project implements a **Machine Learning model** designed to predict the probability of software defects within source code. 

The analysis is performed using **Static Code Metrics**, focusing on two industry-standard frameworks:
* **McCabe’s Cyclomatic Complexity:** Measures the logical complexity and control flow of the program ($v(g)$).
* **Halstead’s Software Science:** Measures computational complexity based on operators and operands (Volume, Effort, Difficulty).

**Goal:** To provide an automated risk assessment for code modules, allowing developers and AI agents to prioritize testing and code reviews.

## 1. Environment Setup and Libraries Injection
In this section, we import the essential Python libraries required for data manipulation, visualization, and machine learning. 
* **Pandas & NumPy:** For efficient data handling and numerical operations.
* **Matplotlib & Seaborn:** For exploratory data analysis and visual representation of data patterns.
* **Scikit-Learn:** For data preprocessing, scaling, and implementing the Logistic Regression model.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

## 2. Data Acquisition
We load the **NASA JM1 dataset**, which contains over 10,000 instances of software modules. Each instance is described by multiple static code metrics. 
At this stage, we also perform an initial check for data integrity and identify missing values (NaN) to ensure the dataset is clean before proceeding to analysis.

In [4]:
train = pd.read_csv('data/archive/jm1.csv')

In [8]:
# Displaying the first few rows to verify successful loading
train.head()

,id,loc,v(g),ev(g),iv(g),n,v,l,d,i,...,lOCode,lOComment,lOBlank,locCodeAndComment,uniq_Op,uniq_Opnd,total_Op,total_Opnd,branchCount,defects
0,1,1.1,1.4,1.4,1.4,1.3,1.30,1.30,1.30,1.30,...,2,2,2,2,1.2,1.2,1.2,1.2,1.4,False
1,2,1.0,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.00,...,1,1,1,1,1,1,1,1,1,True
2,3,72.0,7.0,1.0,6.0,198.0,1134.13,0.05,20.31,55.85,...,51,10,8,1,17,36,112,86,13,True
3,4,190.0,3.0,1.0,3.0,600.0,4348.76,0.06,17.06,254.87,...,129,29,28,2,17,135,329,271,5,True
4,5,37.0,4.0,1.0,4.0,126.0,599.12,0.06,17.19,34.86,...,28,1,6,0,11,16,76,50,7,True


** Use info and describe() on ad_data**

In [10]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 10885 entries, 0 to 10884
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 10885 non-null  int64  
 1   loc                10885 non-null  float64
 2   v(g)               10885 non-null  float64
 3   ev(g)              10885 non-null  float64
 4   iv(g)              10885 non-null  float64
 5   n                  10885 non-null  float64
 6   v                  10885 non-null  float64
 7   l                  10885 non-null  float64
 8   d                  10885 non-null  float64
 9   i                  10885 non-null  float64
 10  e                  10885 non-null  float64
 11  b                  10885 non-null  float64
 12  t                  10885 non-null  float64
 13  lOCode             10885 non-null  int64  
 14  lOComment          10885 non-null  int64  
 15  lOBlank            10885 non-null  int64  
 16  locCodeAndComment  10885 non-null

## 3. Feature Selection: Aligning with Radon Metrics
To ensure the model is practical and compatible with our source code analysis tool (Radon), we narrow down the dataset to only include common metrics. 
This selection focuses on **McCabe's LOC and Cyclomatic Complexity**, alongside **Halstead's core metrics**.

In [ ]:
# 1. הגדרת העמודות שאנחנו רוצים להשאיר (כולל ה-Target)
selected_columns = ['loc', 'v(g)', 'v', 'd', 'e', 'b', 'defects']

# 2. יצירת DataFrame חדש ומצומצם
train = train[selected_columns].copy()

# 3. המרת ה-Target למספרים (0 ו-1)
train['defects'] = train['defects'].astype(int)

# 4. וידוא אחרון שהכל תקין
print("--- Selected Features Info ---")
print(train.info())

--- Selected Features Info ---
<class 'pandas.DataFrame'>
RangeIndex: 10885 entries, 0 to 10884
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   loc      10885 non-null  float64
 1   v(g)     10885 non-null  float64
 2   v        10885 non-null  float64
 3   d        10885 non-null  float64
 4   e        10885 non-null  float64
 5   b        10885 non-null  float64
 6   defects  10885 non-null  int64  
dtypes: float64(6), int64(1)
memory usage: 595.4 KB
None


In [16]:
train.head()

,loc,v(g),v,d,e,b,defects
0,1.1,1.4,1.30,1.30,1.30,1.30,0
1,1.0,1.0,1.00,1.00,1.00,1.00,1
2,72.0,7.0,1134.13,20.31,23029.10,0.38,1
3,190.0,3.0,4348.76,17.06,74202.67,1.45,1
4,37.0,4.0,599.12,17.19,10297.30,0.20,1


# 4. Exploratory Data Analysis (EDA)

## 4.1 Statistical Profiling
In this step, we use `info()` and `describe()` to understand the data types, 
detect missing values, and observe the statistical distribution of our features.

In [17]:
# הצגת הסטטיסטיקה התיאורית
stats = train.describe()
display(stats)

# חישוב אחוז הבאגים בדאטה-סט
bug_percentage = train['defects'].mean() * 100
print(f"\nTarget Distribution: {bug_percentage:.2f}% of modules contain defects.")

,loc,v(g),v,d,e,b,defects
count,10885.000000,10885.000000,10885.000000,10885.000000,1.088500e+04,10885.000000,10885.000000
mean,42.016178,6.348590,673.758017,14.177237,3.683637e+04,0.224766,0.193477
std,76.593332,13.019695,1938.856196,18.709900,4.343678e+05,0.646408,0.395042
min,1.000000,1.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000
25%,11.000000,2.000000,48.430000,3.000000,1.619400e+02,0.020000,0.000000
50%,23.000000,3.000000,217.130000,9.090000,2.031020e+03,0.070000,0.000000
75%,46.000000,7.000000,621.480000,18.900000,1.141643e+04,0.210000,0.000000
max,3442.000000,470.000000,80843.080000,418.200000,3.107978e+07,26.950000,1.000000



Target Distribution: 19.35% of modules contain defects.


### Key Insights from Statistics:
1. **Scale Imbalance:** Features like `Effort (e)` have magnitudes millions of times larger than `Bugs (b)`. Standard scaling is mandatory.
2. **Skewed Data:** High maximum values in `loc` (3442) and `v(g)` (470) compared to their medians suggest the presence of extreme outliers.
3. **Class Imbalance:** Only ~19.4% of the modules are defective. We must account for this during model training using class weighting.